## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-16-18-52-44 +0000,nyt,Judge Sets Spring Retrial for Lindsay Clancy,https://www.nytimes.com/2026/11/16/us/lindsay-...
1,2026-11-16-18-44-00 +0000,wsj,Comex Gold Settles 0.83% Lower at $3847.40,https://www.wsj.com/finance/commodities-future...
2,2026-11-16-18-42-49 +0000,latimes,"California sues OpenAI, alleging it endangered...",https://www.latimes.com/california/story/2026-...
3,2026-11-16-18-41-33 +0000,nyt,Willie Geist Is Leaving ‘Morning Joe’ After Ne...,https://www.nytimes.com/2026/11/16/business/me...
4,2026-11-16-18-39-13 +0000,nyt,Fed Chairman Highlights Economic Uncertainty o...,https://www.nytimes.com/2026/11/16/business/ec...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2564/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
169,trump,29
94,nvidia,24
148,new,20
208,china,15
16,ai,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
45,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...,102
128,2026-11-16-14-46-32 +0000,nypost,Trump ‘would not mind’ Jensen Huang reinstated...,https://nypost.com/2026/11/16/us-news/trump-wo...,91
32,2026-11-16-17-48-36 +0000,nyt,Trump Says He’s Open to Reinstatement of Jense...,https://www.nytimes.com/2026/11/16/us/politics...,89
96,2026-11-16-15-50-59 +0000,nypost,Trump defends Commerce boss Lutnick over Nvidi...,https://nypost.com/2026/11/16/us-news/trump-de...,86
36,2026-11-16-17-43-00 +0000,wsj,Nvidia Scandal Threatens China’s Charm Offensi...,https://www.wsj.com/world/china/nvidia-scandal...,86


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
45,102,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...
306,78,2026-11-15-20-20-57 +0000,nypost,Jensen Huang eyeing comeback as Nvidia CEO for...,https://nypost.com/2026/11/15/business/jensen-...
168,60,2026-11-16-12-03-41 +0000,chitri,Questions swirl around FAA’s new SMART AI syst...,https://www.chicagotribune.com/2026/11/16/faa-...
112,53,2026-11-16-15-07-19 +0000,nypost,Lindsay Clancy to face first-degree murder tri...,https://nypost.com/2026/11/16/us-news/lindsay-...
167,44,2026-11-16-12-04-00 +0000,wsj,A Texas-based U.S. attorney sent what he calle...,https://www.wsj.com/arts-culture/books/the-whi...
152,37,2026-11-16-13-10-23 +0000,cbc,Ukraine launches biggest drone attack on Russi...,https://www.cbc.ca/news/world/ukraine-russia-d...
190,32,2026-11-16-11-00-00 +0000,nypost,NYC uses twice as many delivery contractors as...,https://nypost.com/2026/11/16/business/nyc-use...
285,31,2026-11-15-22-13-00 +0000,wsj,"Miguel Ángel Félix Gallardo, Architect of Mexi...",https://www.wsj.com/world/americas/miguel-ange...
305,30,2026-11-15-20-30-00 +0000,wapo,"Southeast Asian leaders kick-start summit, wit...",https://www.washingtonpost.com/world/2026/11/1...
242,28,2026-11-16-06-01-33 +0000,bbc,'I'd throw up in the car park': How school anx...,https://www.bbc.co.uk/news/articles/cm62pl6lgl...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
